In [6]:
import sys
from pathlib import Path

from collections import Counter
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from src.dataset import (
    build_label_mapping,
    encode_labels,
    load_protein_csv,
)

from src.utils import (
    set_random_seed,
)

set_random_seed(42)

sequences, labels = load_protein_csv("../data/proteins.csv")

label_to_index, index_to_label = build_label_mapping(labels)
encoded_labels = encode_labels(labels, label_to_index)

train_seq, test_cv_seq, train_labels, test_cv_labels = train_test_split(
    sequences,
    encoded_labels,
    test_size=0.4,
    random_state=42,
    stratify=encoded_labels,
)

valid_seq, test_seq, valid_labels, test_labels = train_test_split(
    test_cv_seq,
    test_cv_labels,
    test_size=0.5,
    random_state=42,
    stratify=test_cv_labels,
)

print(f"Training examples: {len(train_seq)}")
print(f"Validation examples: {len(valid_seq)}")
print(f"Test examples: {len(test_seq)}")

print(f"Training label counts: {Counter(train_labels)}")
print(f"Validation label counts: {Counter(valid_labels)}")
print(f"Test label counts: {Counter(test_labels)}")

print(f"Label to Index Mapping: {label_to_index}")

Training examples: 60
Validation examples: 20
Test examples: 20
Training label counts: Counter({2: 15, 3: 15, 0: 15, 1: 15})
Validation label counts: Counter({1: 5, 3: 5, 0: 5, 2: 5})
Test label counts: Counter({3: 5, 1: 5, 0: 5, 2: 5})
Label to Index Mapping: {'dna_binding': 0, 'enzyme': 1, 'membrane': 2, 'structural': 3}


In [5]:
from pathlib import Path
import sys

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader


PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.dataset import (
    ProteinSequenceDataset,
    build_amino_acid_vocab,
    build_label_mapping,
    collate_protein_batch,
    encode_labels,
    load_protein_csv,
)
from src.utils import set_random_seed


set_random_seed(42)

sequences, text_labels = load_protein_csv(
    "../data/proteins.csv"
)

label_to_index, index_to_label = build_label_mapping(
    text_labels
)

encoded_labels = encode_labels(
    text_labels,
    label_to_index,
)

train_seq, test_cv_seq, train_labels, test_cv_labels = train_test_split(
    sequences,
    encoded_labels,
    test_size=0.4,
    random_state=42,
    stratify=encoded_labels,
)

valid_seq, test_seq, valid_labels, test_labels = train_test_split(
    test_cv_seq,
    test_cv_labels,
    test_size=0.5,
    random_state=42,
    stratify=test_cv_labels,
)

vocab = build_amino_acid_vocab()

train_dataset = ProteinSequenceDataset(
    train_seq,
    train_labels,
    vocab,
)

valid_dataset = ProteinSequenceDataset(
    valid_seq,
    valid_labels,
    vocab,
)

test_dataset = ProteinSequenceDataset(
    test_seq,
    test_labels,
    vocab,
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_protein_batch,
)

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_protein_batch,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_protein_batch,
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(valid_loader)}")
print(f"Test batches: {len(test_loader)}")

padded_sequences, sequence_lengths, batch_labels = next(
    iter(train_loader)
)

print("\nOne training batch")

print("\nPadded sequences shape:")
print(padded_sequences.shape)

print("\nSequence lengths:")
print(sequence_lengths)

print("\nBatch labels:")
print(batch_labels)

print("\nBatch labels shape:")
print(batch_labels.shape)

print("\nVocabulary size:")
print(len(vocab))

print("\nNumber of classes:")
print(len(label_to_index))

print("\nLabel mapping:")
print(label_to_index)

Training batches: 8
Validation batches: 3
Test batches: 3

One training batch

Padded sequences shape:
torch.Size([8, 66])

Sequence lengths:
tensor([44, 35, 58, 46, 57, 51, 66, 54])

Batch labels:
tensor([0, 0, 0, 0, 0, 1, 2, 2])

Batch labels shape:
torch.Size([8])

Vocabulary size:
22

Number of classes:
4

Label mapping:
{'dna_binding': 0, 'enzyme': 1, 'membrane': 2, 'structural': 3}
